# 3. Import and export with provenance

**SYNTHETIC TEACHING DATA.** This notebook creates rectangular WGS84 GeoJSON count cells and validates them using the same contract as the viewer. It exports a comparison as JSON and CSV, plus a count-cell GeoJSON for the viewer. It does not import rasters, GeoTIFFs, polygons with holes, overlapping cells, or arbitrary geometries.

All exports retain source, citation, license, year, synthetic status, bounds, and aggregation method. CSV also carries the complete supplied metadata object in `metadata_json`. Keep source notices when moving these files to another application.


In [ ]:
import json
from pathlib import Path
from population_tools import load_demo, import_geojson, compare, export_csv, METHOD
demo = load_demo()
layer = demo['layers'][0]
metadata = dict(layer['metadata'], id='notebook-import', title='Synthetic notebook import')
features = []
for cell in layer['cells']:
    w, s, e, n = cell['bounds']
    features.append({'type': 'Feature', 'geometry': {'type': 'Polygon',
        'coordinates': [[[w,s], [e,s], [e,n], [w,n], [w,s]]]},
        'properties': {'population': cell['population']}})
document = {'type': 'FeatureCollection', 'metadata': metadata, 'features': features}
imported = import_geojson(document)
results = compare([imported], demo['default_aoi'])
assert abs(results[0]['population'] - 400) < 1e-7
print('Validated SYNTHETIC cells:', len(imported['cells']))


In [ ]:
output = Path('outputs')
output.mkdir(exist_ok=True)
(output / 'synthetic-count-cells.geojson').write_text(json.dumps(document, indent=2), encoding='utf-8')
(output / 'synthetic-comparison.json').write_text(json.dumps({
    'notice': 'SYNTHETIC teaching counts; no observed data',
    'bounds': demo['default_aoi'], 'method': METHOD, 'results': results
}, indent=2), encoding='utf-8')
(output / 'synthetic-comparison.csv').write_text(export_csv(results), encoding='utf-8')
print('Created:', ', '.join(p.name for p in sorted(output.iterdir())))
print(export_csv(results))


In JupyterLite, open the `outputs` folder in the file browser and download the files. Import `synthetic-count-cells.geojson` into the standalone viewer. It should give 400 people for the shared example rectangle.

For a real grid, replace the entire metadata record with the actual source's citation, exact release, license, notices, concept, census vintage, and processing history. Set `synthetic` to `false` only for actual observed or modeled source data. This contract validates numeric and geometric structure; it cannot verify the truth of supplied provenance or grant permission to redistribute a dataset.


## Data and software credit

JupyterLite-Population-Data-Comparison contributors (2026). *Invented comparison cells, version 1.0*. CC0-1.0: https://creativecommons.org/publicdomain/zero/1.0/. Generated by deterministic arithmetic, with no census, imagery, boundary, or population data inputs. Full per-layer metadata are in `demo.json`; the repository's `data/README.md` records the checksum. `population_tools.py` calculates spherical area-weighted count fractions and preserves source metadata.

Execution: Project Jupyter's JupyterLite (https://jupyterlite.readthedocs.io/) and Pyodide (https://pyodide.org/). Explanations are rendered with Jupyter Book (https://jupyterbook.org/). These dependencies retain their upstream licenses. The standalone viewer uses Leaflet 1.9.4 (BSD-2-Clause).

RustGIS, GeoLibre, JupyterGIS and a FieldWork widget remain future integration work. This exercise exports supported text/GeoJSON artifacts and does not claim native RustGIS-in-Pyodide or GeoTIFF support.
